### Structured Output

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY")
model=init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",model_provider="openrouter")
model

d:\ai-learning-roadmap\complete_agentic_ai\langchainupdated\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001F53194DFD0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={})

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # BaseModel means we have input validation
    title: str = Field(..., description="The title of the movie") #The ... tells Pydantic: “You must provide this value when creating the model; it cannot be omitted.
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movies rating out of 10")
    genre: str = Field(..., description="The genre of the movie")

In [5]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001F53194DFD0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parame

In [6]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='Here is a comprehensive breakdown of **Christopher Nolan’s *Inception* (2010)**.\n\n---\n\n### **1. Basic Information**\n*   **Director/Writer/Producer:** Christopher Nolan\n*   **Producer:** Emma Thomas\n*   **Cinematography:** Wally Pfister\n*   **Music:** Hans Zimmer\n*   **Editor:** Lee Smith\n*   **Production Companies:** Syncopy Inc., Legendary Pictures\n*   **Distributor:** Warner Bros. Pictures\n*   **Release Date:** July 16, 2010 (London Premiere); July 21, 2010 (US/UK)\n*   **Runtime:** 148 minutes\n*   **Budget:** ~$160 million\n*   **Box Office:** ~$836.8 million worldwide\n*   **Genre:** Science Fiction, Action, Heist Thriller, Psychological Drama\n\n---\n\n### **2. Principal Cast**\n*   **Leonardo DiCaprio** as **Dom Cobb** – A skilled "Extractor" haunted by the memory of his late wife.\n*   **Marion Cotillard** as **Mal Cobb** – Dom’s deceased wife, who manifests as a violent projection in his subconscious.\n*   **Elliot Page** (credited as Ellen Page)

In [7]:
response = model_with_structured_output.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, genre='Sci-Fi/Action/Thriller')

### Message outout alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie") #The ... tells Pydantic: “You must provide this value when creating the model; it cannot be omitted.
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movies rating out of 10")
    genre: str = Field(..., description="The genre of the movie")

model_with_structured_output = model.with_structured_output(Movie, include_raw = True) #to display by deafault how the raw message will come

response = model_with_structured_output.invoke("Provide details about the moview Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for details about the movie "Inception". I need to provide information about this movie. I should use the Movie function to get the details. The function requires title, year, director, rating, and genre. I know Inception is a 2010 movie directed by Christopher Nolan, with a rating of around 8.8/10, and it\'s a science fiction/action thriller. I\'ll call the function with these details.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for details about the movie "Inception". I need to provide information about this movie. I should use the Movie function to get the details. The function requires title, year, director, rating, and genre. I know Inception is a 2010 movie directed by Christopher Nolan, with a rating of around 8.8/10, and it\'s a science fiction/action thriller. I\'ll call the function with these details.'}]}, response_meta

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Tom Berenger', role='Browning'), Actor(name='Dileep Rao', role='Yusuf')], genres=['Science Fiction', 'Action', 'Thriller', 'Heist'], budget=160.0)

### TypeDict

In [13]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """A movie with its details."""
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The release year of the movie"]
    cast: list[dict[str, str]]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_typdict_stucture= model.with_structured_output(MovieDict)

response = model_with_typdict_stucture.invoke("Provide details about the movie Inception")
response


{'title': 'Inception',
 'year': 2010,
 'cast': [{'actor': 'Leonardo DiCaprio', 'character': 'Dom Cobb'},
  {'actor': 'Marion Cotillard', 'character': 'Mal Cobb'},
  {'actor': 'Elliot Page', 'character': 'Ariadne'},
  {'actor': 'Tom Hardy', 'character': 'Eames'},
  {'actor': 'Ken Watanabe', 'character': 'Saito'},
  {'actor': 'Cillian Murphy', 'character': 'Robert Fischer'},
  {'actor': 'Tom Berenger', 'character': 'Browning'},
  {'actor': 'Michael Caine', 'character': 'Professor Stephen Miles'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'budget': 160000000}

In [ ]:
# here input validation is not necessary unlike pydantic, if title is int

from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Michael Caine', 'role': 'Miles'}],
 'genres': ['Science Fiction', 'Action', 'Thriller'],
 'budget': 160000000}

###  Profile

In [ ]:
model_with_typdict_stucture.profile # with structure outputs, we cannot see the profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [17]:
model.profile

{'name': 'Nemotron 3 Ultra (free)',
 'release_date': '2026-06-04',
 'last_updated': '2026-06-04',
 'open_weights': True,
 'max_input_tokens': 1000000,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': False,
 'attachment': False,
 'temperature': True,
 'tool_call_streaming': True}

### Dataclasses

In [ ]:
# starting with pydanctic
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", # here added with openrouter: in start
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result
result["structured_response"]

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='c614f0f6-cb1e-491f-8562-da144dac6f09'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to extract contact info from the given string: "John Doe, john@example.com, (555) 123-4567". I need to parse the name, email, and phone number. The format appears to be: name, email, phone. So I should extract "John Doe" as name, "john@example.com" as email, "(555) 123-4567" as phone. Then call the ContactInfo function with those parameters. Let me do that.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to extract contact info from the given string: "John Doe, john@example.com, (555) 123-4567". I need to parse the name, email, and phone number. The format appears to be: name, email, phone. So I should extract "John Doe" as name, "john@example.com" a

In [21]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", # here added with openrouter: in start
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [22]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", # here added with openrouter: in start
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

TooManyRequestsResponseError: Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day